In [1]:
# 04_train_black_box_models.ipynb
# Black-box ML models – Software Defect Prediction
# Random Forest, Gradient Boosting, SVM, MLP

import sys
import time
import psutil
import os
sys.path.append("../src")
from utils import DATASETS, TARGET, RANDOM_STATE, N_SPLITS


import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import copy

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score
)

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

processed_path = Path("../datasets/processed")
results_path = Path("../results/black_box_models")
results_path.mkdir(parents=True, exist_ok=True)

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

def evaluate_fold(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0)
    }

models = {
    "Random_Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "Gradient_Boosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=1.0,
        random_state=RANDOM_STATE
    ),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(
            kernel="rbf",
            probability=True,
            random_state=RANDOM_STATE
        ))
    ]),
    "MLP": Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPClassifier(
            hidden_layer_sizes=(100,),
            max_iter=500,
            random_state=RANDOM_STATE
        ))
    ])
}

results = []
training_times = []

for ds in DATASETS:
    print(f"\nDATASET: {ds}")

    dataset_path = results_path / ds
    dataset_path.mkdir(parents=True, exist_ok=True)

    data = pd.read_csv(processed_path / f"{ds}_processed.csv")
    X = data.drop(columns=[TARGET])
    y = data[TARGET]

    for model_name, model in models.items():
        print(f" → {model_name}")

        model_path = dataset_path / model_name
        model_path.mkdir(parents=True, exist_ok=True)

        fold_metrics = []
        best_model = copy.deepcopy(model)  
        best_f1 = -1

        for fold, (train_idx, test_idx) in enumerate(cv.split(X, y)):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            smote = SMOTE(random_state=RANDOM_STATE)
            resampled = smote.fit_resample(X_train, y_train)
            X_train_res, y_train_res = resampled[0], resampled[1]

            fold_model = copy.deepcopy(model)
            _t0 = time.perf_counter()
            fold_model.fit(X_train_res, y_train_res)
            _fold_train_time = time.perf_counter() - _t0
            y_pred = fold_model.predict(X_test)

            fm = evaluate_fold(y_test, y_pred)
            fm["train_time_s"] = _fold_train_time
            fold_metrics.append(fm)

            if fm["F1-score"] > best_f1:
                best_f1 = fm["F1-score"]
                best_model = fold_model

        avg_metrics: dict = {
            "Dataset": ds,
            "Model": model_name,
            "Accuracy": round(float(np.mean([m["Accuracy"] for m in fold_metrics])), 4),
            "Precision": round(float(np.mean([m["Precision"] for m in fold_metrics])), 4),
            "Recall": round(float(np.mean([m["Recall"] for m in fold_metrics])), 4),
            "F1-score": round(float(np.mean([m["F1-score"] for m in fold_metrics])), 4),
            "Train_Time_Mean_s": round(float(np.mean([m["train_time_s"] for m in fold_metrics])), 4),
            "Train_Time_Std_s":  round(float(np.std( [m["train_time_s"] for m in fold_metrics])), 4)
        }
        results.append(avg_metrics)

        training_times.append({
            "Dataset": ds, "Model": model_name,
            "Train_Time_Mean_s": avg_metrics["Train_Time_Mean_s"],
            "Train_Time_Std_s":  avg_metrics["Train_Time_Std_s"]
        })

        joblib.dump(best_model, model_path / "model.joblib")

results_df = pd.DataFrame(results)
results_df = results_df[
    ["Dataset", "Model", "Accuracy", "Precision", "Recall", "F1-score",
     "Train_Time_Mean_s", "Train_Time_Std_s"]
]

timing_df = pd.DataFrame(training_times)
timing_df.to_csv(results_path / "black_box_models_training_times.csv", index=False)
print("\nTraining times (s):")
print(timing_df.to_string(index=False))

results_df.to_csv(
    results_path / "black_box_models_metrics.csv",
    index=False
)

results_df



DATASET: CM1
 → Random_Forest
 → Gradient_Boosting
 → SVM
 → MLP


/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/e


DATASET: JM1
 → Random_Forest
 → Gradient_Boosting
 → SVM
 → MLP


/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(



DATASET: KC1
 → Random_Forest
 → Gradient_Boosting
 → SVM
 → MLP


...

/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/e


DATASET: PC1
 → Random_Forest
 → Gradient_Boosting
 → SVM
 → MLP


/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/e


Training times (s):
Dataset             Model  Train_Time_Mean_s  Train_Time_Std_s
    CM1     Random_Forest             0.9831            0.1906
    CM1 Gradient_Boosting             1.8333            0.2759
    CM1               SVM             0.1936            0.0364
    CM1               MLP             5.1319            3.0291
    JM1     Random_Forest             6.3609            0.6383
    JM1 Gradient_Boosting            23.0006            3.8843
    JM1               SVM            90.8173           11.0526
    JM1               MLP            39.6967           16.3724
    KC1     Random_Forest             1.6164            0.3988
    KC1 Gradient_Boosting             4.1157            0.3705
    KC1               SVM             2.0841            1.0063
    KC1               MLP             7.3321            1.6282
    KC2     Random_Forest             0.8518            0.1582
    KC2 Gradient_Boosting             1.8580            0.4712
    KC2               SVM         

,Dataset,Model,Accuracy,Precision,Recall,F1-score,Train_Time_Mean_s,Train_Time_Std_s
0,CM1,Random_Forest,0.8420,0.2869,0.2689,0.2711,0.9831,0.1906
1,CM1,Gradient_Boosting,0.8397,0.3042,0.3111,0.2990,1.8333,0.2759
2,CM1,SVM,0.7698,0.2660,0.6222,0.3707,0.1936,0.0364
3,CM1,MLP,0.8645,0.4020,0.5178,0.4515,5.1319,3.0291
4,JM1,Random_Forest,0.7964,0.3738,0.2503,0.2995,6.3609,0.6383
5,JM1,Gradient_Boosting,0.7728,0.3059,0.2386,0.2676,23.0006,3.8843
6,JM1,SVM,0.6676,0.2721,0.5422,0.3620,90.8173,11.0526
7,JM1,MLP,0.6937,0.2862,0.5032,0.3647,39.6967,16.3724
8,KC1,Random_Forest,0.7849,0.3337,0.2884,0.3067,1.6164,0.3988
9,KC1,Gradient_Boosting,0.7818,0.3158,0.2605,0.2845,4.1157,0.3705
